# V46｜共享 AE 128 维联合优化 + SelfExpr + GAT/GeneRead（全 Count）

**仅运行 V46**，不重训 V44 / V45；所有训练只使用 dropout40 Count，`true_data.csv` 仅在最后评价时读取。

改变 V45 的**唯一核心机制**：先使用双视图共享 AE (512→128) 构建 SelfExpr 图，随后**冻结浅层 512 维编码器及 SelfExpr 图**；在 300 轮 GAT + GeneRead 训练中，**同步更新 AE 瓶颈映射 512→128、AE 原解码器以及 GAT+GeneRead**。

- 不添加 OwnOnly、不添加双频分支、不另开训练阶段、不额外增加遮盖比例。
- AE 预训练为 Count；后端恢复为 Count；评估为原始 Count。
- 后端输入在每轮**先隐藏原 Count 观测非零的 15%**，再编码、解码、取邻居表达；预测目标只在隐藏位置计算 MSE。
- 训练后的 128 维特征在推断时由**同一套已经微调的 AE**重新编码生成，不直接重用原始输入的缓存特征。
- 分块计算训练误差，同时采用双分支梯度汇集，确保 AE 的后半部分确实收到梯度。
- 用于自检的 6% 留出位置在 AE 预训练和构图**之前**清零，绝不使用真实完整数据参与训练或模型选择。


## 原始读取、工具函数与数据预处理（原样保留）


In [ ]:
# 运行环境预检查：缺少依赖时在这里给出明确提示。
import importlib.util, sys, os
_required = ['numpy', 'pandas', 'scipy', 'sklearn', 'torch', 'torch_geometric']
_missing = [name for name in _required if importlib.util.find_spec(name) is None]
if _missing:
    raise ImportError(
        '缺少依赖: ' + ', '.join(_missing) + '\n'
        '请在当前 Notebook 所用的 Python 环境中安装缺失包；torch_geometric 需按本机 PyTorch/CUDA 版本选择对应安装命令。'
    )
print('依赖检查通过。Python:', sys.version.split()[0], '| PyTorch:', __import__('torch').__version__)


### 导库

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.neighbors import NearestNeighbors

from torch_geometric.nn import GCNConv
import pandas as pd
from sklearn.metrics import pairwise_distances
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import contingency_matrix
import warnings
import sys
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
# for cosine similarity metric used in evaluation
from sklearn.metrics.pairwise import cosine_similarity
from functools import partial
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
from sklearn.metrics.cluster import contingency_matrix
import math
import random
import copy
from sklearn.neighbors import NearestNeighbors
from torch_geometric.nn import GATConv

### 工具函数

In [ ]:
#加载数据
def load_data(data_path):
    data_csv = pd.read_csv(data_path, index_col=0)
    cells = data_csv.columns.values
    genes = data_csv.index.values
    data = data_csv.values.T


    return data, cells, genes

In [ ]:
#数据处理
def take_norm(data, cellwise_norm=True, log1p=True):
    data_norm = data.copy()
    data_norm = data_norm.astype('float32')
    if cellwise_norm:
        libs = data.sum(axis=1)
        norm_factor = np.diag(np.median(libs) / libs)
        data_norm = np.dot(norm_factor, data_norm)

    if log1p:
        data_norm = np.log2(data_norm + 1.)
    return data_norm

In [ ]:
#模拟dropout
def impute_dropout(X, seed=None, drop_rate=0.1):
    """
    X: original testing set
    ========
    returns:
    X_zero: copy of X with zeros
    """

    X_zero = np.copy(X)
    i, j = np.nonzero(X_zero)
    if seed is not None:
        np.random.seed(seed)

    ix = np.random.choice(range(len(i)), int(
        np.floor(drop_rate * len(i))), replace=False)
    X_zero[i[ix], j[ix]] = 0.0

    return X_zero

In [ ]:
def purity_score(y_true, y_pred):
    # compute contingency matrix (also called confusion matrix)
    contingency_matrix1 = contingency_matrix(y_true, y_pred)
    # return purity
    return np.sum(np.amax(contingency_matrix1, axis=0)) / np.sum(contingency_matrix1) 
def JaccardInd(ytrue,ypred):
    n = len(ytrue)
    a,b,c,d = 0,0,0,0
    for i in range(n-1):
        for j in range(i+1,n):
            if ((ypred[i] == ypred[j])&(ytrue[i]==ytrue[j])):
                a = a + 1
            elif ((ypred[i] == ypred[j])&(ytrue[i]!=ytrue[j])):
                b = b + 1
            elif ((ypred[i] != ypred[j])&(ytrue[i]==ytrue[j])):
                c = c + 1
            else:
                d = d + 1
    if (a==0)&(b==0)&(c==0):
        return 0
    else:
        return a/(a+b+c)

In [ ]:
#评估指标
def cos_simility(imputed_data, original_data):
    # 使用 sklearn 的 cosine_similarity
    return np.mean(cosine_similarity(original_data, imputed_data))

def l1_distance(imputed_data, original_data):

    return np.mean(np.abs(original_data-imputed_data))

def RMSE(imputed_data, original_data):
    return np.sqrt(np.mean((original_data - imputed_data)**2))
 # 计算两个向量person相关系数
def pearson_corr(imputed_data, original_data):
    Y = original_data
    fake_Y = imputed_data
    fake_Y, Y = fake_Y.reshape(-1), Y.reshape(-1)
    fake_Y_mean, Y_mean = np.mean(fake_Y), np.mean(Y)
    corr = (np.sum((fake_Y - fake_Y_mean) * (Y - Y_mean))) / (
            np.sqrt(np.sum((fake_Y - fake_Y_mean) ** 2)) * np.sqrt(np.sum((Y - Y_mean) ** 2)))
    return corr

### 输入 dropout40（训练阶段不读取真实完整矩阵）

In [ ]:
#定义数据集名称和dropout率
# dataset_name = 'Zeisel'
# drop_rate = 0.4
device='cpu'

In [ ]:
# 与原数据读取函数 load_data() 保持一致；但训练阶段物理隔离 true_data.csv。
from pathlib import Path
_DATA_DIR_CANDIDATES = [Path(r'E:/code/my/zeisel_data'), Path.cwd(), Path.cwd()/'zeisel_data', Path('/mnt/data')]
def _find_dataset_file(filename):
    for directory in _DATA_DIR_CANDIDATES:
        p=directory/filename
        if p.is_file():return p.resolve()
    raise FileNotFoundError(filename+' 未找到；请放在 E:/code/my/zeisel_data 或 Notebook 目录')
DROPOUT_PATH=_find_dataset_file('Zeisel_dropout_40_gene_cell.csv')
DATA_DIR=DROPOUT_PATH.parent
drop_data, cells, genes=load_data(str(DROPOUT_PATH))
print('V38 TRAIN ONLY dropout40:',DROPOUT_PATH,'shape',drop_data.shape)
# 注意：故意不定义 groundTruth_data、reference_counts、Y、TRUE_PATH。


In [ ]:
print('训练输入:',drop_data.shape,'细胞:',len(cells),'基因:',len(genes))

In [ ]:
# # 模拟dropout事件
# drop_data = impute_dropout(groundTruth_data, drop_rate=drop_rate)
# print('dataset: {}, drop rate: {}'.format(dataset_name, drop_rate))

In [ ]:
# 训练中仅保留稀疏输入 X，不创建真实参考目标 Y。
X=torch.FloatTensor(np.copy(drop_data))

In [ ]:
X_input=X.clone().to(device)  # V45: 仅原始 Count，不调用 take_norm/log1p
n_cells,n_genes=X_input.shape
print('cells:',n_cells,'genes:',n_genes,'| 全 Count')

## 训练配置与通用函数（Count 尺度）

In [ ]:
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from IPython.display import display
from contextlib import contextmanager
import hashlib
import gc
import math
import random
import time

# V46: AE=300、SelfExpr=200、GAT+GeneRead=300；AE后半段与图模块联合训练，无 OwnOnly。
CONFIG = dict(
    seed=42, deterministic=False, model_device='auto', cpu_threads=1,
    ae_epochs=300, ae_lr=1e-3, ae_hidden=512, ae_latent=128,
    ae_dropout=0.1, ae_batch_size=256, ae_mask_ratio=0.15,
    cl_weight=0.1, cl_temperature=0.2, cl_warmup=50, cl_ramp_epochs=50,
    graph_k=15, selfexpr_l1=0.05, selfexpr_l2=0.01,
    selfexpr_max_steps=200, selfexpr_min_steps=50,
    selfexpr_tol=1e-5, selfexpr_edge_epsilon=1e-6,
    
    loss='mse', feature_dim=128, gat_heads=4,
    attention_dropout=0.1, sk_hidden=32, mask_ratio=0.15,
    gate_bound=0.1, gate_anchor_weight=0.02,
    weak_zero_weight=0.05, weak_zero_sample_ratio=0.5,
    pair_block=32768, cell_block=256, gene_block=512,
    grad_clip=5.0, log_every=10,
)
RUN_MODE = 'full'  # 一键正式运行，不需要手动切换模式。
if RUN_MODE == 'smoke':
    CONFIG.update(ae_epochs=3, cl_warmup=0, cl_ramp_epochs=1,
                  selfexpr_max_steps=3, selfexpr_min_steps=2,
                  branch_epochs=2, fusion_epochs=2)
elif RUN_MODE != 'full':
    raise ValueError("RUN_MODE 必须是 full 或 smoke")

ENCODER_VARIANTS = (
    dict(name='OneView_AE_Count', views=1),
    dict(name='TwoView_SharedAE_Count_V26Graph', views=2),
)

def expression_digest(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256(str((a.shape, str(a.dtype))).encode())
    h.update(a.tobytes())
    return h.hexdigest()

def graph_digest(graph):
    w = graph.tocsr(copy=True); w.sort_indices()
    h = hashlib.sha256(str(w.shape).encode())
    for a in (w.indptr, w.indices, w.data):
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()

@contextmanager
def controlled_rng(seed, device):
    # 同一个主 seed 派生的固定随机流。退出时恢复外部 RNG，绝不是多 seed 实验。
    dev = torch.device(device)
    devices = [dev.index if dev.index is not None else torch.cuda.current_device()] if dev.type == 'cuda' else []
    with torch.random.fork_rng(devices=devices):
        torch.random.default_generator.manual_seed(int(seed))
        if dev.type == 'cuda':
            with torch.cuda.device(dev):
                torch.cuda.manual_seed(int(seed))
        yield

@contextmanager
def comparison_cpu_scope(device, cfg):
    old = torch.get_num_threads()
    if torch.device(device).type == 'cpu':
        torch.set_num_threads(int(cfg['cpu_threads']))
    try:
        yield
    finally:
        torch.set_num_threads(old)

torch.use_deterministic_algorithms(bool(CONFIG['deterministic']))
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
model_device = torch.device(('cuda' if torch.cuda.is_available() else 'cpu')
                            if CONFIG['model_device'] == 'auto' else CONFIG['model_device'])
count_array = np.asarray(drop_data)
if count_array.ndim != 2 or min(count_array.shape) < 2:
    raise ValueError('dropout count 必须为细胞×基因矩阵，至少两个细胞和两个基因')
if not np.isfinite(count_array).all():
    raise ValueError('count 输入包含 NaN/Inf')
if (count_array < 0).any():
    raise ValueError('输入必须非负')
if not (count_array > 0).any() or (count_array > 0).all():
    raise ValueError('需要同时存在观测非零和未知零')
X_count = torch.as_tensor(np.ascontiguousarray(count_array, dtype=np.float32), device=model_device)
X_graph = X_count  # V46 所有表达矩阵、AE 重构和训练监督均使用 Count。
if not bool(torch.isfinite(X_count).all()):
    raise ValueError('count 超出 float32 范围')
_v20_input_hash = expression_digest(count_array)
_v20_tensor_hash = expression_digest(X_count.detach().cpu().numpy())
print('V46：仅运行联合微调方案；全 Count，冻结 512 浅层，优化 128 瓶颈+AE 解码器+GAT/GeneRead。')
print('主 seed=42 | 设备:', model_device, '| 细胞×基因:', count_array.shape, '| 模式:', RUN_MODE)
print('本版不自动重跑单视图或完整 V14。')
print('V46 ONLY: graph_AE=count | downstream=count | final=count')

In [ ]:
import gc


import math


import random


import time


import numpy as np


import pandas as pd


from scipy import sparse


import torch


from torch import nn


from torch.nn import functional as F


from torch_geometric.nn import GATv2Conv


def sync_device(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def slices(size, block):
    for start in range(0, size, int(block)):
        yield slice(start, min(size, start+int(block)))


def seed_recovery(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def prepare_recovery_graph(graph, raw, device):
    """W[i,j] is a message from j to i. Graph/evidence use only dropout input."""
    w = graph.tocsr(copy=True).astype(np.float32)
    if w.shape != (raw.shape[0], raw.shape[0]):
        raise ValueError('图的细胞数与原dropout输入不一致')
    if w.diagonal().any():
        w.setdiag(0)
    w.eliminate_zeros(); w.sort_indices()
    if not np.isfinite(w.data).all() or (w.data < 0).any():
        raise ValueError('SelfExpr权重必须有限且非负')
    sums = np.asarray(w.sum(axis=1)).ravel()
    inv = np.divide(1., sums, out=np.zeros_like(sums), where=sums > 0)
    w = (sparse.diags(inv) @ w).tocsr()
    coo = w.tocoo()
    ij = torch.as_tensor(np.stack([coo.row, coo.col]), dtype=torch.long, device=device)
    values = torch.as_tensor(coo.data, device=device)
    graph_t = dict(
        W=torch.sparse_coo_tensor(ij, values, w.shape, device=device).coalesce(),
        edge_index=torch.stack([ij[1], ij[0]]),
        edge_attr=torch.log(values.clamp_min(1e-8))[:, None],
        has_neighbor=torch.as_tensor(np.asarray(w.getnnz(axis=1)).ravel() > 0, device=device),
    )
    return graph_t


@torch.no_grad()
def recovery_evidence(corrupted, graph, gene_block=512, working_scale="count"):
    """Recompute from masked input. Hidden values never enter neighbor statistics."""
    support = torch.empty_like(corrupted)
    disagreement = torch.empty_like(corrupted)
    for gs in slices(corrupted.shape[1], gene_block):
        values = corrupted[:, gs]
        nonzero = (values > 0).to(values.dtype)
        log_values = values if working_scale == "log" else torch.log1p(values)
        p = torch.sparse.mm(graph['W'], nonzero).clamp(0., 1.)
        denominator = p.clamp_min(1e-8)
        mean = torch.sparse.mm(graph['W'], log_values)/denominator
        second = torch.sparse.mm(graph['W'], log_values.square())/denominator
        sd = (second-mean.square()).clamp_min(0.).sqrt()
        sd = torch.where(p > 1e-8, sd, torch.zeros_like(sd))
        support[:, gs] = p
        disagreement[:, gs] = sd/(1.+sd)
    return dict(support=support, disagreement=disagreement)


def sample_recovery_task(raw, zero_pool, cfg):
    """Fresh mask of observed nonzeros; untouched input zeros remain unlabeled."""
    observed = raw > 0
    hidden = observed & (torch.rand_like(raw) < float(cfg['mask_ratio']))
    if not bool(hidden.any()):
        # 小型数据的兜底；仍只遮盖一个确实观测非零的位置。
        first = observed.nonzero(as_tuple=False)[0]
        hidden[first[0], first[1]] = True
    positive = hidden.nonzero(as_tuple=False)
    corrupted = raw.masked_fill(hidden, 0.)
    negative_count = min(zero_pool.numel(), int(math.ceil(len(positive)*cfg['weak_zero_sample_ratio'])))
    if negative_count and cfg['weak_zero_weight'] > 0.:
        # 均匀有放回抽样，避免每轮对整个零证据池进行昂贵的randperm。
        chosen = zero_pool[torch.randint(zero_pool.numel(), (negative_count,), device=raw.device)]
        negative = torch.stack([chosen//raw.shape[1], chosen%raw.shape[1]], dim=1)
    else:
        negative = torch.empty((0,2), dtype=torch.long, device=raw.device)
    pairs = torch.cat([positive, negative], dim=0)
    targets = torch.cat([raw[positive[:,0], positive[:,1]], raw.new_zeros(len(negative))])
    # positive/weak zero分开取均值。总目标不是把所有原始零都当0重构。
    weights = torch.cat([
        raw.new_full((len(positive),), 1./len(positive)),
        raw.new_full((len(negative),), cfg['weak_zero_weight']/max(1,len(negative))),
    ])
    return dict(corrupted=corrupted, hidden=hidden, pairs=pairs, targets=targets,
                weights=weights, positive_count=len(positive), weak_zero_count=len(negative))


def preserve_observed(prediction, raw):
    # 保持 raw 的原始精度；只替换 dropout 输入的零位置。
    x = np.asarray(raw)
    result = np.asarray(prediction, dtype=np.float64).copy()
    result[x > 0] = x[x > 0]
    if not np.array_equal(result[x > 0], x[x > 0]):
        raise AssertionError('观测值未保留')
    return result


def pearson_flat(a, b):
    a = np.asarray(a, dtype=np.float64).ravel()
    b = np.asarray(b, dtype=np.float64).ravel()
    if a.size < 2 or a.size != b.size:
        return np.nan
    ac, bc = a-a.mean(), b-b.mean()
    den = np.sqrt(np.dot(ac, ac))*np.sqrt(np.dot(bc, bc))
    return float(np.dot(ac, bc)/den) if den > 0 else np.nan


def metric_values(prediction, truth, mask=None):
    if mask is None:
        p, y = np.asarray(prediction).ravel(), np.asarray(truth).ravel()
    else:
        p, y = np.asarray(prediction)[mask], np.asarray(truth)[mask]
    p, y = p.astype(np.float64), y.astype(np.float64)
    if not p.size:
        return dict(n=0, PCC=np.nan, MSE=np.nan, RMSE=np.nan, L1=np.nan,
                    bias=np.nan, pred_mean=np.nan, truth_mean=np.nan)
    error = p-y
    mse = float(np.mean(error*error))
    return dict(n=int(p.size), PCC=pearson_flat(p, y), MSE=mse, RMSE=float(np.sqrt(mse)),
                L1=float(np.abs(error).mean()), bias=float(error.mean()),
                pred_mean=float(p.mean()), truth_mean=float(y.mean()))


def frozen_model_hash(model):
    """Stable digest used to verify that the frozen own predictor is unchanged."""
    import hashlib
    h = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        h.update(name.encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()



## SelfExpr：保持 V44 的非负稀疏自表达与 Mutual Top-15

In [ ]:
def row_normalize(a):
    a=a.tocsr(copy=True).astype(np.float32)
    a.eliminate_zeros();a.sort_indices()
    if not np.isfinite(a.data).all() or (a.data<0).any():
        raise ValueError('图权重必须非负且有限')
    sums=np.asarray(a.sum(axis=1)).ravel()
    inv=np.divide(1.,sums,out=np.zeros_like(sums),where=sums>0)
    return (sparse.diags(inv)@a).tocsr()


def topk_matrix(a,k,include_self=False):
    """仅筛选矩阵元素；行循环不是细胞/基因分批训练。并列权重按列号稳定排序。"""
    n=a.shape[0]
    k=min(int(k),n if include_self else n-1)
    if k<1:raise ValueError('邻居数必须为正')
    is_sparse=sparse.issparse(a)
    if is_sparse:a=a.tocsr(copy=True);a.eliminate_zeros()
    rows=[];cols=[];values=[]
    for i in range(n):
        if is_sparse:
            start,stop=a.indptr[i:i+2];j=a.indices[start:stop];v=a.data[start:stop]
        else:
            j=np.arange(n);v=np.asarray(a[i])
        keep=(v>0)&np.isfinite(v)
        if not include_self:keep&=(j!=i)
        j=j[keep];v=v[keep]
        chosen=np.lexsort((j,-v))[:k]
        rows.extend([i]*len(chosen));cols.extend(j[chosen]);values.extend(v[chosen])
    return sparse.csr_matrix((values,(rows,cols)),shape=(n,n),dtype=np.float32)


def mutual_topk(a,k):
    directed=topk_matrix(a,k)
    topology=directed.copy();topology.data[:]=1
    mutual=topology.multiply(topology.T)
    symmetric=((directed+directed.T)*.5).multiply(mutual).tocsr()
    return row_normalize(symmetric)


def graph_stats(w):
    degree=np.asarray(w.getnnz(axis=1)).ravel()
    components,_=connected_components(w,directed=False)
    return dict(edges=w.nnz//2,mean_neighbors=float(degree.mean()),
                max_neighbors=int(degree.max()),isolated=int((degree==0).sum()),
                components=int(components))


@torch.no_grad()
def learn_self_expression(processed,valid,cfg):
    """全细胞候选的非负稀疏自表达，整矩阵近端加速梯度；无 KNN 预筛选。"""
    if processed.ndim!=2:raise ValueError('processed 必须为细胞×特征矩阵')
    n,g=processed.shape
    l1=float(cfg['selfexpr_l1']);l2=float(cfg['selfexpr_l2'])
    if l1<0 or l2<0:raise ValueError('正则强度不能为负')
    if int(cfg['selfexpr_max_steps'])<1:raise ValueError('selfexpr_max_steps 必须为正')
    allowed=torch.as_tensor(valid,dtype=processed.dtype,device=processed.device)
    C=torch.zeros((n,n),dtype=processed.dtype,device=processed.device)
    extrapolated=C.clone()
    momentum=1.0
    # ||Z||_F²/G >= ||Z||_2²/G，使用保守的 Lipschitz 上界，避免估计误差。
    lipschitz=float(processed.square().sum().item()/g)+l2
    if lipschitz<=0:raise ValueError('构图矩阵不能全为零')
    step=1.0/lipschitz
    rows=[];last_logged=0
    for iteration in range(1,int(cfg['selfexpr_max_steps'])+1):
        residual=extrapolated@processed-processed
        gradient=(residual@processed.T)/g+l2*extrapolated
        # 非负 L1 近端算子；diag(C)=0，禁止细胞直接复制自己。
        new_C=(extrapolated-step*gradient-step*l1).clamp_min_(0)
        new_C.fill_diagonal_(0)
        new_C.mul_(allowed[:,None]).mul_(allowed[None,:])
        delta=new_C-C
        relative_change=float((torch.linalg.vector_norm(delta)/
            torch.linalg.vector_norm(C).clamp_min(1e-8)).item())
        next_momentum=(1+math.sqrt(1+4*momentum*momentum))/2
        extrapolated=new_C+((momentum-1)/next_momentum)*delta
        C=new_C;momentum=next_momentum
        del residual,gradient,new_C,delta
        should_stop=iteration>=cfg['selfexpr_min_steps'] and relative_change<cfg['selfexpr_tol']
        if iteration==1 or iteration%cfg['log_every']==0 or iteration==cfg['selfexpr_max_steps'] or should_stop:
            error=C@processed-processed
            mse=float(error.square().mean().item())
            objective=0.5*mse+l1*float(C.sum().item())/n+0.5*l2*float(C.square().sum().item())/n
            fraction=float((C>cfg['selfexpr_edge_epsilon']).float().sum().item()/max(n*(n-1),1))
            rows.append(dict(step=iteration,feature_reconstruction_MSE=mse,
                             objective_per_cell=objective,relative_change=relative_change,
                             nonzero_coefficient_fraction=fraction))
            print('稀疏自表达 %d/%d | feature MSE %.6f | objective %.6f | 相对变化 %.3g'%(
                iteration,cfg['selfexpr_max_steps'],mse,objective,relative_change))
            last_logged=iteration
            del error
        if should_stop:break
    converged=relative_change<cfg['selfexpr_tol'] and iteration>=cfg['selfexpr_min_steps']
    print('自表达停止:',('达到相对变化阈值' if converged else '达到迭代上限，未达到变化阈值'))
    # 先对称化，再仅保留最强 k 项；不把 N×N 的稠密系数全部转换为稀疏边。
    symmetric=(C+C.T)*0.5
    k=min(int(cfg['graph_k']),n-1)
    if k<1:raise ValueError('graph_k 必须为正')
    values,cols=torch.topk(symmetric,k=k,dim=1)
    rows_tensor=torch.arange(n,device=processed.device)[:,None].expand_as(cols)
    keep=values>cfg['selfexpr_edge_epsilon']
    affinity=sparse.csr_matrix((values[keep].cpu().numpy(),
        (rows_tensor[keep].cpu().numpy(),cols[keep].cpu().numpy())),shape=(n,n))
    graph=mutual_topk(affinity,cfg['graph_k'])
    history=pd.DataFrame(rows)
    return graph,history,dict(iterations=iteration,converged=converged,
                             l1=l1,l2=l2,step_size=step),None

## 双视图共享 AE：512 维构图，128 维复用，所有损失和重构均为 Count

In [ ]:
class CountGraphAE(nn.Module):
    """Count AE：512 浅层构图、128 瓶颈复用；输出和 MSE 均为 Count。"""
    def __init__(self, raw, cfg):
        super().__init__()
        genes = raw.shape[1]
        h, d = int(cfg['ae_hidden']), int(cfg['ae_latent'])
        self.input_transform = 'identity'  # 上游完成工作尺度转换，不允许在 AE 内部二次 log。
        self.shallow = nn.Sequential(nn.Linear(genes, h), nn.LayerNorm(h), nn.ReLU())
        self.deep = nn.Sequential(nn.Dropout(float(cfg['ae_dropout'])), nn.Linear(h, d))
        self.decoder = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, genes))
        # gene_scale、输出和目标使用同一个工作尺度（count 或 log）。
        self.register_buffer('gene_scale', raw.detach().mean(dim=0).clamp_min(.1))
        nn.init.normal_(self.decoder[-1].weight, std=.01)
        nn.init.constant_(self.decoder[-1].bias, math.log(math.expm1(1.)))

    def transform_encoder_input(self, x):
        # 输入已经是相应工作域；不再额外执行 log1p。
        return x

    def forward(self, x):
        h = self.shallow(self.transform_encoder_input(x))
        # 解码器输出同工作尺度；损失不会出现 log 与 count 相减。
        return F.softplus(self.decoder(self.deep(h)))*self.gene_scale, h


def shallow_contrast_loss(h1, h2, temperature):
    """同细胞跨视图为正；其他细胞的同视图与跨视图表示均为负。"""
    if h1.ndim != 2 or h1.shape != h2.shape or len(h1) < 2:
        raise ValueError('两路特征须同形，且至少包含两个有效细胞')
    if temperature <= 0:
        raise ValueError('temperature 必须大于 0')
    n = h1.shape[0]
    a = F.normalize(h1, dim=1, eps=1e-8)
    b = F.normalize(h2, dim=1, eps=1e-8)
    z = torch.cat([a, b], dim=0)
    similarity = z @ z.T
    self_mask = torch.eye(2*n, dtype=torch.bool, device=z.device)
    logits = (similarity / float(temperature)).masked_fill(self_mask, float('-inf'))
    anchors = torch.arange(2*n, device=z.device, dtype=torch.long)
    positive_ids = (anchors+n) % (2*n)
    loss = F.cross_entropy(logits, positive_ids)
    with torch.no_grad():
        positive = similarity[anchors, positive_ids].mean()
        negative_mask = ~self_mask
        negative_mask[anchors, positive_ids] = False
        negative = similarity[negative_mask].mean()
        accuracy = (logits.argmax(dim=1) == positive_ids).float().mean()
    return loss, dict(positive_cosine=float(positive), negative_cosine=float(negative),
                      pair_accuracy=float(accuracy))


def contrast_batches(order, block):
    # 最后一批若只有一个细胞，合并到前一批，避免没有负样本。
    parts = list(order.split(int(block)))
    if len(parts) > 1 and len(parts[-1]) == 1:
        parts[-2] = torch.cat([parts[-2], parts[-1]])
        parts.pop()
    return parts


def make_ae_views(target, cfg):
    """两路独立非零遮盖；同一套 AE 参数。输入和目标均为原始 Count。"""
    observed = target > 0
    mask1 = observed & (torch.rand_like(target) < float(cfg['ae_mask_ratio']))
    mask2 = observed & (torch.rand_like(target) < float(cfg['ae_mask_ratio']))
    return target.masked_fill(mask1,0.), target.masked_fill(mask2,0.)


@torch.no_grad()
def shallow_blocks(models, x, cfg):
    blocks = []
    for model in models:
        model.eval()
        blocks.append(torch.cat([model.shallow(model.transform_encoder_input(x[cs]))
                                  for cs in slices(len(x), cfg['ae_batch_size'])]))
    return blocks


@torch.no_grad()
def fit_shallow_graph_transform(blocks, valid):
    # 学到的浅层特征做中心化和整块 RMS 缩放，控制 SelfExpr 正则的尺度。
    # 这一步作用于隐藏特征，不是对输入 count 做库大小归一化或 log。
    transforms, arrays, diagnostics = [], [], []
    for index, h in enumerate(blocks):
        center = h[valid].mean(dim=0, keepdim=True)
        energy = (h[valid] - center).square().mean()
        if not bool(torch.isfinite(energy)) or float(energy) < 1e-12:
            raise FloatingPointError('浅层特征接近塌缩，停止构图')
        scale = energy.sqrt()
        value = (h-center)/scale
        value[~valid] = 0.
        arrays.append(value)
        transforms.append(dict(center=center.detach(), scale=scale.detach()))
        diagnostics.append(dict(encoder=index+1, width=h.shape[1],
                                centered_MSE=float(energy),
                                mean_feature_std=float(h[valid].std(dim=0, unbiased=False).mean())))
    return torch.cat(arrays, dim=1).detach(), transforms, pd.DataFrame(diagnostics)


def validate_encoder_cfg(cfg):
    for key in ('ae_epochs', 'ae_hidden', 'ae_latent', 'ae_batch_size', 'cl_ramp_epochs', 'cpu_threads'):
        if int(cfg[key]) < 1:
            raise ValueError(key + ' 必须为正整数')
    if cfg['ae_batch_size'] < 2 or cfg['ae_hidden'] < 2:
        raise ValueError('ae_batch_size 和 ae_hidden 至少为 2')
    if not 0. <= cfg['ae_mask_ratio'] < 1. or not 0. <= cfg['ae_dropout'] < 1.:
        raise ValueError('AE 遮盖率和隐藏层 dropout 必须在 [0,1)')
    if cfg.get('ae_view_mode', 'A_baseline_mask_mask') not in (
            'A_baseline_mask_mask',):
        raise ValueError('ae_view_mode 无效')
    if cfg.get('working_scale', 'count') != 'count':
        raise ValueError('V43 的下游恢复必须为 count；构图 AE 另行使用 log1p')
    if cfg['cl_weight'] < 0. or cfg['cl_temperature'] <= 0. or cfg['cl_warmup'] < 0:
        raise ValueError('对比权重/温度/预热配置无效')


def v38_ae_reconstruction(prediction, target, masked_view, cfg):
    """AE reconstruction WITHOUT interpreting original unknown zeros as observed negatives in B/C.
    No reference/ground truth is available. Both masks and targets derived from current dropout40 fit input.
    """
    mode=cfg.get('ae_loss_mode','A_full')
    if mode=='A_full':
        return F.mse_loss(prediction,target)
    obs=target>0
    # 在 V39 抽稀视图中，50 -> 30 或 50 -> 0 都需要恢复；不只监督变 0 的位置。
    masked=obs & (masked_view < target - 1e-6)
    keep=obs & (~masked)
    if not bool(masked.any()):
        # Rare degenerate tiny batch; do not treat original zero as a negative.
        if not bool(obs.any()):
            return prediction.sum()*0.
        masked=obs
    recovery=(prediction[masked]-target[masked]).square().mean()
    if mode=='B_masked':
        return recovery
    if mode=='C_masked_observed':
        observed=(prediction[keep]-target[keep]).square().mean() if bool(keep.any()) else prediction.sum()*0.
        return recovery+float(cfg.get('ae_observed_weight',0.15))*observed
    raise ValueError('Unknown V38 AE loss mode: '+str(mode))


def train_shallow_encoders(x, spec, cfg):
    validate_encoder_cfg(cfg)
    if spec['views'] not in (1, 2):
        raise ValueError('只支持单视图 AE 与双视图共享 AE')
    valid = (x > 0).any(dim=1)
    indices = torch.arange(len(x), device=x.device, dtype=torch.long)
    if int(valid.sum()) < 2:
        raise ValueError('至少需要两个有效细胞参与构图比较')
    with controlled_rng(cfg['seed'], x.device):
        models = nn.ModuleList([CountGraphAE(x, cfg).to(x.device)])
    # 每组只有一套 AE；双视图前向调用同一对象，编码器/解码器全部共享。
    model = models[0]
    initial_hashes = tuple(frozen_model_hash(m) for m in models)
    # 输入 x 已是本组指定尺度，AE 预测与目标全部在该尺度比较。
    # 每组单独计算自身工作域的梯度归一化常数；不跨尺度比较 AE 损失值。
    mse_scale = float((x.double().square().mean() if cfg.get('ae_loss_mode','A_full')=='A_full' else x[x>0].double().square().mean()).clamp_min(1.).cpu())
    if not math.isfinite(mse_scale):
        raise FloatingPointError('AE MSE 缩放标量非有限')
    optimizer = torch.optim.Adam(models.parameters(), lr=cfg['ae_lr'])
    history = []
    sync_device(x.device); started = time.perf_counter()
    for epoch in range(1, int(cfg['ae_epochs'])+1):
        models.train(); optimizer.zero_grad(set_to_none=True)
        ramp = min(1., max(0., (epoch-int(cfg['cl_warmup'])) / int(cfg['cl_ramp_epochs'])))
        weight = float(cfg['cl_weight']) * ramp if spec['views'] == 2 else 0.
        totals = dict(reconstruction_MSE=0., reconstruction_MSE_scaled=0., contrast_loss=0., total_loss=0.,
                      positive_cosine=0., negative_cosine=0., pair_accuracy=0.)
        sync_device(x.device); tick = time.perf_counter()
        # 每轮覆盖全部细胞和基因（含原始零），累积梯度后更新一次。
        with controlled_rng(int(cfg['seed'])*1000003+epoch, x.device):
            order = indices[torch.randperm(len(indices), device=x.device)]
            for ids in contrast_batches(order, cfg['ae_batch_size']):
                target = x[ids]
                # 分离随机流：额外视图随机数不会改变另一组的细胞顺序和首路隐藏 Dropout。
                batch_id = int(ids[0])
                base_stream = int(cfg['seed'])*1000003 + epoch*100000 + batch_id*10
                if spec['views'] == 1:
                    with controlled_rng(base_stream+2, x.device):
                        p1, h1 = model(target)
                    rec_raw = v38_ae_reconstruction(p1, target, target, cfg)
                    cl = rec_raw.new_zeros(())
                    diagnostics = dict(positive_cosine=0., negative_cosine=0., pair_accuracy=0.)
                else:
                    with controlled_rng(base_stream+1, x.device):
                        a, b = make_ae_views(target, cfg)
                    with controlled_rng(base_stream+2, x.device):
                        p1, h1 = model(a)
                    with controlled_rng(base_stream+3, x.device):
                        p2, h2 = model(b)
                    rec_raw = .5*(v38_ae_reconstruction(p1,target,a,cfg) + v38_ae_reconstruction(p2,target,b,cfg))
                    contrast_valid = (target > 0).any(dim=1)
                    if int(contrast_valid.sum()) >= 2:
                        cl, diagnostics = shallow_contrast_loss(
                            h1[contrast_valid], h2[contrast_valid], cfg['cl_temperature'])
                    else:
                        cl = rec_raw.new_zeros(())
                        diagnostics = dict(positive_cosine=0., negative_cosine=0., pair_accuracy=0.)
                rec = rec_raw/mse_scale
                loss = rec + weight*cl
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError('AE/对比损失非有限')
                fraction = len(ids)/len(indices)
                (loss*fraction).backward()
                for key, value in dict(reconstruction_MSE=float(rec_raw.detach()),
                                       reconstruction_MSE_scaled=float(rec.detach()), contrast_loss=float(cl.detach()),
                                       total_loss=float(loss.detach()), **diagnostics).items():
                    totals[key] += fraction*value
        torch.nn.utils.clip_grad_norm_(models.parameters(), cfg['grad_clip'], error_if_nonfinite=True)
        optimizer.step()
        sync_device(x.device); seconds = time.perf_counter()-tick
        history.append(dict(variant=spec['name'], seed=cfg['seed'], epoch=epoch,
                            views=spec['views'], MSE_scale=mse_scale,
                            contrast_weight=weight, epoch_seconds=seconds, **totals))
        if epoch == 1 or epoch % cfg['log_every'] == 0 or epoch == cfg['ae_epochs']:
            print(f'{spec["name"]} AE {epoch}/{cfg["ae_epochs"]} | count MSE {totals["reconstruction_MSE"]:.4f}'
                  f' | MSE/scale {totals["reconstruction_MSE_scaled"]:.6f}'
                  f' | CL {totals["contrast_loss"]:.4f} × {weight:.3f} | {seconds:.2f}s/轮')
    optimizer.zero_grad(set_to_none=True)
    models.eval().requires_grad_(False)
    features, transforms, feature_df = fit_shallow_graph_transform(shallow_blocks(models, x, cfg), valid)
    feature_df.insert(0, 'variant', spec['name'])
    meta = dict(variant=spec['name'], stage='graph_AE', input_transform=cfg.get('graph_ae_scale','count'), epochs=cfg['ae_epochs'],
                encoders=1, views=spec['views'], MSE_scale=mse_scale,
                parameters=sum(p.numel() for p in models.parameters()),
                graph_feature_width=features.shape[1], train_seconds=time.perf_counter()-started,
                initial_encoder_hashes=initial_hashes)
    del optimizer
    return models, features, transforms, pd.DataFrame(history), feature_df, meta


def matched_recovery_task(raw, zero_pool, cfg, stream, epoch):
    # 保证不同图的边数、弱零抽样数和 GAT dropout 不改变下一轮的正例遮盖。
    with controlled_rng(int(cfg['seed'])*1000003 + int(stream)*10000 + int(epoch), raw.device):
        return sample_recovery_task(raw, zero_pool, cfg)


In [ ]:
def graph_for_context(features, valid, raw, cfg):
    if bool(valid.any()) and bool(features.square().sum() > 0):
        b, history, status, _ = learn_self_expression(features, valid.detach().cpu().numpy(), cfg)
    else:
        b = sparse.csr_matrix((len(raw), len(raw)), dtype=np.float32)
        history = pd.DataFrame()
        status = dict(iterations=0, converged=False)
    return b, prepare_recovery_graph(b, raw, raw.device), history, status



## V46：共享 AE 后半部分可联合训练；浅层 512 维和 SelfExpr 图保持固定

In [ ]:
class SharedAECountBackbone(nn.Module):
    """共享 AE：浅层 512 固定构图；瓶颈 128 与解码器在后端联合微调。"""
    def __init__(self, ae):
        super().__init__()
        self.ae = ae.eval()
        self.n_genes = int(self.ae.decoder[-1].out_features)
        # gene_scale 原本就是 Count 预训练中注册的缓冲区，不重复训练。
        self.register_buffer('gene_scale', self.ae.gene_scale.detach().clone())
        if int(self.ae.deep[-1].out_features) != 128:
            raise ValueError('预期瓶颈层为 128 维')
        self.freeze_shallow_only()

    def freeze_shallow_only(self):
        self.ae.shallow.requires_grad_(False)
        self.ae.deep.requires_grad_(True)
        self.ae.decoder.requires_grad_(True)
        self.ae.eval()  # 关闭 AE 的 Dropout；训练和推断使用一致的确定性表示。
        return self

    def encode(self, count):
        # 人工遮盖由调用方在编码前完成。shallow 只做无梯度前向，SelfExpr 所依赖权重不变。
        with torch.no_grad():
            shallow = self.ae.shallow(count)
        return self.ae.deep(shallow)

    def decode_matrix(self, latent):
        return F.softplus(self.ae.decoder(latent))*self.gene_scale


def v46_prepare_base(raw, cfg):
    assert cfg['working_scale'] == cfg['graph_ae_scale'] == 'count'
    spec = dict(name='V46_TwoView_SharedAE_Count', views=2)
    encoders, features, transforms, ae_history, feature_df, ae_meta = train_shallow_encoders(raw, spec, cfg)
    graph, graph_t, graph_history, graph_status = graph_for_context(
        features, (raw > 0).any(1), raw, cfg)
    backbone = SharedAECountBackbone(encoders[0]).to(raw.device).eval()
    assert int(features.shape[1]) == int(cfg['ae_hidden']) == 512
    with torch.no_grad():
        z = backbone.encode(raw)
        assert tuple(z.shape) == (len(raw), 128)
        basic = backbone.decode_matrix(z)
        assert basic.shape == raw.shape and torch.isfinite(basic).all()
    print('共享 AE：固定 512 浅层 → SelfExpr；128 瓶颈/原解码器 → 与 GAT+GeneRead 联合优化')
    print('SelfExpr:', graph_status)
    del encoders, features, transforms, graph_t
    return dict(backbone=backbone, graph=graph, graph_hash=graph_digest(graph),
                ae_history=ae_history, ae_meta=ae_meta,
                graph_history=graph_history, graph_status=graph_status)


## V46：GAT + GeneRead 使用 AE 128 维，并与瓶颈层/解码器联合优化（保持原模块）

In [ ]:
def v31_neighbor_index(graph, raw):
    """Row i lists OTHER cells sending to i; padding never counts as an observation."""
    w = graph.tocsr(copy=True).astype(np.float32)
    if w.shape != (len(raw), len(raw)):
        raise ValueError('Graph/input shape mismatch')
    w.setdiag(0); w.eliminate_zeros(); w.sort_indices()
    if not np.isfinite(w.data).all() or (w.data < 0).any():
        raise ValueError('Invalid graph weights')
    degree = np.diff(w.indptr)
    width = max(1, int(degree.max()))
    ids = np.zeros((len(raw), width), dtype=np.int64)
    valid = np.zeros((len(raw), width), dtype=bool)
    weights = np.zeros((len(raw), width), dtype=np.float32)
    for i in range(len(raw)):
        lo, hi = w.indptr[i:i+2]; k = hi-lo
        if k:
            ids[i, :k] = w.indices[lo:hi]
            valid[i, :k] = True
            weights[i, :k] = w.data[lo:hi] / w.data[lo:hi].sum()
    return dict(index=torch.as_tensor(ids, device=raw.device),
                valid=torch.as_tensor(valid, device=raw.device),
                weight=torch.as_tensor(weights, device=raw.device))


@torch.no_grad()
def v31_read_context(corrupted, neighbors, working_scale='count'):
    """同工作尺度取邻居表达；计数模式允许深度比校正，log 模式不对 log 值做 count 乘法。"""
    library = corrupted.sum(1)
    positive = library[library > 0]
    median = positive.median() if len(positive) else library.new_tensor(1.)
    depth = torch.where(library > 0, library / median.clamp_min(1e-8),
                        torch.ones_like(library)).clamp_min(1e-8)
    return dict(counts=corrupted, depth=depth, neighbors=neighbors, working_scale=working_scale)


class V31GATCorrection(nn.Module):
    """GAT + GeneRead 图辅助 Count 残差模型（128 维特征）。"""
    def __init__(self, own, cfg, use_read=False):
        super().__init__()
        self.use_read = bool(use_read)
        dim = int(cfg['feature_dim']); heads = int(cfg['gat_heads'])
        if dim % heads:
            raise ValueError('feature_dim must be divisible by gat_heads')
        self.gat = GATv2Conv(dim, dim//heads, heads=heads, concat=True,
                            dropout=float(cfg['attention_dropout']),
                            add_self_loops=True, edge_dim=1)
        self.simple_graph_scale = nn.Parameter(torch.zeros(1))
        gd = int(cfg['gene_embedding']); hidden = int(cfg['decoder_hidden'])
        self.gene_embedding = nn.Embedding(own.n_genes, gd)
        self.residual_decoder = nn.Sequential(
            nn.Linear(dim+gd+1, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))
        nn.init.normal_(self.gene_embedding.weight, std=.02)
        nn.init.zeros_(self.residual_decoder[-1].weight)
        nn.init.zeros_(self.residual_decoder[-1].bias)
        if self.use_read:
            rd = int(cfg['read_dim'])
            self.read_query = nn.Linear(dim, rd, bias=False)
            self.read_gene = nn.Linear(gd, rd, bias=False)
            self.read_key = nn.Linear(dim, rd, bias=False)
            self.read_norm = math.sqrt(rd)
            # 7 evidence features + the original decoder inputs.
            self.read_gate = nn.Sequential(nn.Linear(dim+gd+1+7, int(cfg['read_hidden'])),
                                           nn.GELU(), nn.Linear(int(cfg['read_hidden']), 1))
            nn.init.zeros_(self.read_gate[-1].weight)
            nn.init.zeros_(self.read_gate[-1].bias)

    def fuse_nodes(self, h, graph_t):
        if graph_t['edge_index'].numel():
            g = self.gat(h, graph_t['edge_index'], edge_attr=graph_t['edge_attr'])
        else:
            g = torch.zeros_like(h)
        return h + self.simple_graph_scale * g

    def read_pairs(self, fused, pairs, gene_features, scale, context):
        ci, gi = pairs[:, 0], pairs[:, 1]
        neighbors = context['neighbors']
        js = neighbors['index'][ci]
        valid = neighbors['valid'][ci]
        weight = neighbors['weight'][ci]
        values = context['counts'][js, gi[:, None]]
        observed = valid & (values > 0)
        positive_count = observed.sum(1).to(fused.dtype)
        has_observation = positive_count > 0
        support = (weight * observed.to(weight.dtype)).sum(1)

        # Gene-conditioned weights, restricted to observed neighbor values.
        query = torch.tanh(self.read_query(fused[ci]) + self.read_gene(gene_features))
        keys = self.read_key(fused)
        logits = (query[:, None, :] * keys[js]).sum(-1) / self.read_norm
        logits = logits + weight.clamp_min(1e-12).log()
        logits = logits.masked_fill(~observed, -torch.finfo(logits.dtype).max)
        # All-missing rows remain finite: softmax, then mask and renormalize.
        attention = torch.softmax(logits, dim=1) * observed.to(logits.dtype)
        attention = attention / attention.sum(1, keepdim=True).clamp_min(1e-8)

        # Convert neighbor counts to the query cell's observed library-depth scale.
        # Raw model input and count targets are unchanged. No log value is added to count.
        if context['working_scale'] == 'count':
            values = values * context['depth'][ci, None] / context['depth'][js]
        # log 模式 values 本身就是 log1p(count)，不能乘 count 尺度的库大小比。
        values = torch.where(observed, values, torch.zeros_like(values))
        mean = (attention * values).sum(1)
        variance = (attention * (values-mean[:, None]).square()).sum(1)
        std = (variance.clamp_min(0.) + 1e-8).sqrt()
        maximum = values.max(1)[0]
        fraction = positive_count / valid.sum(1).clamp_min(1).to(fused.dtype)
        features = torch.stack((torch.log1p(mean/scale), torch.log1p(std/scale),
                                support, fraction, torch.log1p(positive_count),
                                torch.log1p(maximum/scale),
                                torch.log1p(context['depth'][ci])), dim=1)
        return mean, features, has_observation, support

    def decode_pairs(self, fused, pairs, base, gene_scale, context=None):
        ci, gi = pairs[:, 0], pairs[:, 1]
        scale = gene_scale[gi].clamp_min(.1)
        gene = self.gene_embedding(gi)
        inputs = torch.cat((fused[ci], gene, torch.log1p(base.clamp_min(0.)/scale)[:, None]), 1)
        delta = self.residual_decoder(inputs).squeeze(-1) * scale
        gate = torch.zeros_like(base)
        support = torch.zeros_like(base)
        has_observation = torch.zeros_like(base, dtype=torch.bool)
        if self.use_read:
            if context is None:
                raise ValueError('The gene reader requires the current masked input')
            mean, features, has_observation, support = self.read_pairs(fused, pairs, gene, scale, context)
            # A signed residual gain, NOT a dropout probability or a zero label.
            gate = torch.tanh(self.read_gate(torch.cat((inputs, features), 1)).squeeze(-1))
            gate = torch.where(has_observation, gate, torch.zeros_like(gate))
            delta = delta + gate * (mean-base)
        prediction = (base+delta).clamp_min(0.)
        return prediction, delta, dict(gate=gate, support=support, has_observation=has_observation)


def v31_mask_task(raw, cfg, epoch):
    # Same random stream as the old GAT baseline; mask only observed positives.
    with controlled_rng(int(cfg['seed'])*1000003 + 260000 + int(epoch), raw.device):
        hidden = (raw > 0) & (torch.rand_like(raw) < float(cfg['mask_ratio']))
        if not bool(hidden.any()):
            first = (raw > 0).nonzero(as_tuple=False)[0]
            hidden[first[0], first[1]] = True
        pairs = hidden.nonzero(as_tuple=False)
        target = raw[pairs[:, 0], pairs[:, 1]].detach()
        corrupted = raw.masked_fill(hidden, 0.)
    return dict(corrupted=corrupted, pairs=pairs, targets=target)




## 原 V32 检出支持度筛选（算法不变）

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from scipy import sparse
import numpy as np
import pandas as pd

def v32_binary(raw):
    x = np.asarray(raw)
    if x.ndim != 2 or not np.isfinite(x).all() or (x < 0).any():
        raise ValueError('需要非负且有限的细胞×基因 count')
    return sparse.csr_matrix((x > 0).astype(np.float32))

def v32_cell_evidence_graph(binary, cfg):
    """从当前可见检出记录独立建图；不看目标值、不含自环、不根据目标基因挑选邻居。"""
    n, g = binary.shape
    if n < 3 or g < 3:
        raise ValueError('至少需要 3 个细胞与 3 个基因')
    weighted = TfidfTransformer(norm='l2').fit_transform(binary)
    dims = min(int(cfg['svd_components']), n - 1, g - 1)
    features = TruncatedSVD(n_components=dims, random_state=int(cfg['seed'])).fit_transform(weighted)
    features = normalize(features, norm='l2')
    k = min(int(cfg['cell_neighbors']), n - 1)
    # n_neighbors=k+1，一定剔除自身，必要时多取以防重复/近零向量导致自身位置不在首位。
    search_n = min(n, k+8)
    nn = NearestNeighbors(n_neighbors=search_n, metric='cosine', algorithm='brute', n_jobs=1)
    nn.fit(features)
    dist, ids = nn.kneighbors(features)
    rows, cols, vals = [], [], []
    for i in range(n):
        idx = ids[i][ids[i] != i][:k]
        if len(idx) == 0:
            continue
        chosen_dist = np.maximum(0., np.minimum(2., dist[i][ids[i] != i][:k]))
        logits = (1. - chosen_dist) / float(cfg['neighbor_temperature'])
        w = np.exp(logits - logits.max())
        w = w / w.sum()
        rows.extend([i]*len(idx))
        cols.extend(idx.tolist())
        vals.extend(w.tolist())
    graph = sparse.csr_matrix((np.array(vals,dtype=np.float32),(rows,cols)),shape=(n,n))
    graph.setdiag(0)
    graph.eliminate_zeros()
    norm = np.asarray(graph.sum(axis=1)).ravel()
    graph = sparse.diags(1./np.maximum(norm,1e-12)) @ graph
    assert graph.diagonal().sum() == 0
    return graph.tocsr()

def v32_gene_coexpression(binary, cfg):
    """二值检出 Phi 相关性，仅保留共检出足够多的正关联基因；禁止目标基因自关联。"""
    n, g = binary.shape
    count = np.asarray(binary.sum(axis=0), dtype=np.float64).ravel()
    detect_p = count / n
    cross = (binary.T @ binary).toarray().astype(np.float64)
    denom = np.sqrt(np.maximum(detect_p*(1.-detect_p),0.))
    denom = denom[:,None]*denom[None,:]
    phi = (cross/n - detect_p[:,None]*detect_p[None,:]) / np.maximum(denom,1e-10)
    enough = ((count >= float(cfg['min_gene_detections'])) &
              (count <= (n-float(cfg['min_gene_detections']))))
    phi[(cross < int(cfg['min_gene_coobserved'])) | (~enough[:,None]) | (~enough[None,:])] = 0.
    phi[~np.isfinite(phi)] = 0.
    np.fill_diagonal(phi,0.)
    phi = np.maximum(phi,0.)
    k = min(int(cfg['gene_neighbors']),g-1)
    rows, cols, weights = [],[],[]
    for gi in range(g):
        valid = np.flatnonzero(phi[gi] > 0)
        if not len(valid):
            continue
        if len(valid) > k:
            valid=valid[np.argpartition(phi[gi,valid],-k)[-k:]]
        w=phi[gi,valid].copy()
        w=w / w.sum()
        rows.extend([gi]*len(valid))
        cols.extend(valid.tolist())
        weights.extend(w.tolist())
    rel=sparse.csr_matrix((np.array(weights,dtype=np.float32),(rows,cols)),shape=(g,g))
    rel.setdiag(0)
    rel.eliminate_zeros()
    assert rel.diagonal().sum() == 0
    return rel

def v32_compute_support(raw,cfg):
    B=v32_binary(raw)
    cell_graph=v32_cell_evidence_graph(B,cfg)
    gene_graph=v32_gene_coexpression(B,cfg)
    # W_cell @ B：每个邻居按预先学习的图权重计入分母；零邻居不会被丢弃。
    neighbor_support=(cell_graph @ B).toarray().astype(np.float32)
    # B @ W_gene.T：同一细胞中与目标基因关联的其他基因的加权检出比例。
    gene_support=(B @ gene_graph.T).toarray().astype(np.float32)
    neighbor_support=np.clip(neighbor_support,0.,1.)
    gene_support=np.clip(gene_support,0.,1.)
    return dict(neighbor=neighbor_support, gene=gene_support,
                joint=np.minimum(neighbor_support,gene_support),
                graph_edges=int(cell_graph.nnz),gene_edges=int(gene_graph.nnz),
                gene_without_positive_links=int(np.sum(np.diff(gene_graph.indptr)==0)),
                detected_per_cell=np.asarray(B.sum(axis=1)).ravel().astype(np.int64))

def v32_quantile_lower(v, q):
    v=np.asarray(v,dtype=np.float64)
    if len(v)==0:
        raise ValueError('未找到用于校准的人工隐藏正例')
    try:
        return float(np.quantile(v,q,method='lower'))
    except TypeError:
        return float(np.quantile(v,q,interpolation='lower'))

def v32_calibrate(score, hidden, detect_counts, cfg):
    """使用人工留出的已知非零正例选择分数阈值，不读取原始零的负标签。"""
    cell, gene=np.nonzero(hidden)
    values=np.asarray(score[cell,gene],dtype=np.float64)
    recall_target=float(cfg['positive_recall_target'])
    q=1.-recall_target
    global_threshold=v32_quantile_lower(values,q)
    groups=int(cfg['depth_groups'])
    edges=np.unique(np.quantile(detect_counts, np.linspace(0.,1.,groups+1)))
    group_ids=np.searchsorted(edges[1:-1], detect_counts, side='right')
    group_thresholds={}
    positive_counts=np.bincount(group_ids[cell],minlength=max(1,len(edges)-1))
    for grp in range(max(1,len(edges)-1)):
        local=values[group_ids[cell]==grp]
        group_thresholds[grp]=(v32_quantile_lower(local,q)
             if len(local)>=int(cfg['min_group_calibration_positives']) else global_threshold)
    cell_counts=np.bincount(cell,minlength=score.shape[0])
    threshold=np.array([group_thresholds[int(g)] for g in group_ids],dtype=np.float32)
    for i in np.flatnonzero(cell_counts>=int(cfg['min_cell_calibration_positives'])):
        threshold[i]=v32_quantile_lower(values[cell==i],q)
    recall=float(np.mean(score[cell,gene] >= threshold[cell]))
    return dict(threshold=threshold,global_threshold=global_threshold,
        calibration_recall=recall, calibrating_positives=int(len(cell)),
        per_cell_threshold_count=int(np.sum(cell_counts>=int(cfg['min_cell_calibration_positives']))),
        depth_bin_edges=edges,group_thresholds=group_thresholds,group_ids=group_ids,
        group_positive_counts=positive_counts)

def v32_apply_threshold(calibration, current_counts):
    edges=calibration['depth_bin_edges']
    groups=np.searchsorted(edges[1:-1],current_counts,side='right')
    thresholds=np.array([calibration['group_thresholds'][int(g)] for g in groups],dtype=np.float32)
    # 每个细胞有足够多留出样本的情况，保留其单独的校准阈值。
    custom_count=int(calibration['per_cell_threshold_count'])
    # 由原校准阶段保留精确的阈值及 cell 是否单独校准标志。
    personal=calibration.get('personal_mask')
    if personal is not None:
        thresholds[personal]=calibration['threshold'][personal]
    return thresholds

def v32_build_filter(raw,cfg):
    """固定一次随机人工隐藏：先隐藏，再在隐藏后的可见输入上重新建证据图与基因图。"""
    original=np.asarray(raw,dtype=np.float32)
    observed=original>0
    rng=np.random.default_rng(int(cfg['seed'])+32000)
    hidden=observed & (rng.random(original.shape) < float(cfg['holdout_fraction']))
    if not hidden.any():
        raise ValueError('没有抽到可用的人工隐藏正例')
    masked=original.copy()
    masked[hidden]=0.
    assert not np.any(masked[hidden]>0)
    masked_hash=expression_digest(masked)
    calibration_support=v32_compute_support(masked,cfg)
    if expression_digest(masked)!=masked_hash:
        raise AssertionError('校准输入被意外改写')
    calibration={}
    for name in ('neighbor','joint'):
        cal=v32_calibrate(calibration_support[name],hidden,calibration_support['detected_per_cell'],cfg)
        cell_counts=np.bincount(np.nonzero(hidden)[0],minlength=len(original))
        cal['personal_mask']=cell_counts>=int(cfg['min_cell_calibration_positives'])
        calibration[name]=cal
    del masked,calibration_support
    inference=v32_compute_support(original,cfg)
    masks={}
    for name in ('neighbor','joint'):
        th=v32_apply_threshold(calibration[name],inference['detected_per_cell'])
        masks[name]=(inference[name]>=th[:,None]) & ~observed
        calibration[name]['inference_threshold']=th
    report=[]
    for name in ('neighbor','joint'):
        cal=calibration[name]
        chosen=masks[name]
        report.append(dict(filter=name,
            target_positive_recall=cfg['positive_recall_target'],
            masked_positive_recall=cal['calibration_recall'],
            masked_positive_count=cal['calibrating_positives'],
            zero_selected=int(chosen.sum()),
            zero_total=int((~observed).sum()),
            zero_selected_fraction=float(chosen.sum()/max(1,(~observed).sum())),
            thresholds_equal_zero=int(np.sum(cal['inference_threshold']<=0)),
            personal_cell_thresholds=cal['per_cell_threshold_count'],
            global_threshold=cal['global_threshold']))
    print('Calibration evidence graph re-estimated AFTER hiding known nonzeros.')
    print('Inference graph edges:',inference['graph_edges'],
          '| gene associations:',inference['gene_edges'],
          '| genes with no positive neighbors:',inference['gene_without_positive_links'])
    return dict(masks=masks,calibration=calibration,report=pd.DataFrame(report),
                evidence=inference,holdout_positive_count=int(hidden.sum()))

def v32_filter_prediction(raw, base, selected):
    raw=np.asarray(raw)
    base=np.asarray(base)
    if raw.shape!=base.shape or selected.shape!=raw.shape:
        raise ValueError('掩码或预测矩阵形状不一致')
    zero=raw==0
    out=raw.copy()
    out[zero & selected]=np.maximum(0.,base[zero & selected])
    assert np.array_equal(out[raw>0],raw[raw>0])
    return out



## 仅 V46：冻结浅层和图，联合微调瓶颈层 + AE 解码器 + GAT/GeneRead

In [ ]:
V46_CONFIG = dict(CONFIG)
V46_CONFIG.update(
    graph_ae_scale='count', working_scale='count',
    ae_epochs=300, ae_loss_mode='C_masked_observed',
    ae_view_mode='A_baseline_mask_mask', ae_observed_weight=0.15,
    correction_epochs=300, correction_lr=3e-4,
    # 微调时小学习率保护预训练的 Count 表示；SelfExpr 图与 512 浅层不会再变。
    bottleneck_lr=1e-5, decoder_lr=3e-5,
    feature_dim=128, gat_heads=4, gene_embedding=16,
    decoder_hidden=128, pair_batch=8192,
    read_dim=32, read_hidden=64, log_every=25
)
V46_FILTER = dict(seed=42, holdout_fraction=.08, positive_recall_target=.95,
    cell_neighbors=15, svd_components=64, neighbor_temperature=.20,
    gene_neighbors=20, min_gene_detections=8, min_gene_coobserved=6,
    min_cell_calibration_positives=40, depth_groups=5,
    min_group_calibration_positives=80)
V46_MODE = 'full'  # 调试可改 smoke；正式一键运行保持 full。
if V46_MODE == 'smoke':
    V46_CONFIG.update(ae_epochs=2, selfexpr_max_steps=3, selfexpr_min_steps=2,
                      correction_epochs=2, cl_warmup=0, cl_ramp_epochs=1, log_every=1)
print('V46 ONLY | 全 Count | 双视图 AE',V46_CONFIG['ae_epochs'],
      '| OwnOnly 0 | 瓶颈+解码器+GAT/GeneRead 联合训练',V46_CONFIG['correction_epochs'])


@torch.no_grad()
def v46_predict(raw, backbone, model, graph_t, neighbors, cfg):
    backbone.eval(); model.eval()
    z = backbone.encode(raw)                            # [cells,128]；只计算一次
    base_matrix = backbone.decode_matrix(z)             # 已预训练 AE 的 Count 解码器
    fused = model.fuse_nodes(z, graph_t)
    context = v31_read_context(raw, neighbors, 'count')
    result = raw.detach().cpu().numpy().copy()
    pairs = (raw == 0).nonzero(as_tuple=False)
    for ps in slices(len(pairs), int(cfg['pair_batch'])):
        sub = pairs[ps]
        base = base_matrix[sub[:,0], sub[:,1]]
        prediction, _, _ = model.decode_pairs(fused,sub,base,backbone.gene_scale,context)
        result[sub[:,0].cpu().numpy(),sub[:,1].cpu().numpy()] = prediction.cpu().numpy()
    return preserve_observed(result,raw.detach().cpu().numpy())



def v46_train_joint_reader(raw, cfg, base):
    """不增加新遮盖：每 epoch 仅做 V45 原有 15% 非零人工遮盖，联合更新 AE 后半段。

    采用分块梯度汇集：把 GAT 计算图及 AE 解码器计算图各算一次，按 pair
    分块在叶子上累积梯度，然后再分别传播至 GAT/AE/deep，避免逐块 retain_graph=True。
    """
    backbone = base['backbone'].freeze_shallow_only()
    graph_t = prepare_recovery_graph(base['graph'], raw, raw.device)
    neighbors = v31_neighbor_index(base['graph'], raw)
    with controlled_rng(int(cfg['seed'])+2600, raw.device):
        model = V31GATCorrection(backbone, cfg, use_read=True).to(raw.device)

    # AE 浅层绝不进入优化器；同一套 AE 保留编码与解码，避免第二个 OwnOnly。
    params = [
        {'params': list(model.parameters()), 'lr': float(cfg['correction_lr'])},
        {'params': list(backbone.ae.deep.parameters()), 'lr': float(cfg['bottleneck_lr'])},
        {'params': list(backbone.ae.decoder.parameters()), 'lr': float(cfg['decoder_lr'])},
    ]
    optimizer = torch.optim.Adam(params)
    trainable = list(model.parameters()) + list(backbone.ae.deep.parameters()) + list(backbone.ae.decoder.parameters())
    original_shallow = frozen_model_hash(backbone.ae.shallow)
    original_deep = frozen_model_hash(backbone.ae.deep)
    original_decoder = frozen_model_hash(backbone.ae.decoder)
    loss_scale = float(raw[raw > 0].double().square().mean().clamp_min(1.).cpu())
    history = []

    with comparison_cpu_scope(raw.device, cfg):
        for epoch in range(1, int(cfg['correction_epochs'])+1):
            task = v31_mask_task(raw, cfg, epoch)
            corrupted, pairs, targets = task['corrupted'], task['pairs'], task['targets']
            # 必须先遮盖，再输入固定浅层。不能预计算完整 raw 的 128 维做监督。
            assert not bool((corrupted[pairs[:, 0], pairs[:, 1]] > 0).any())
            context = v31_read_context(corrupted, neighbors, 'count')
            optimizer.zero_grad(set_to_none=True)
            backbone.ae.eval()   # 关闭 deep dropout，避免训练/推断特征分布不一致。
            model.train()

            # z 保留自动求导，误差最终能传回 deep。浅层在 encode 内部 no_grad。
            z = backbone.encode(corrupted)
            # 两个轻量梯度桥接叶子，避免每块展开整张 AE/GAT 计算图。
            z_bridge = z.detach().requires_grad_(True)
            base_matrix = backbone.decode_matrix(z_bridge)
            base_leaf = base_matrix.detach().requires_grad_(True)
            with controlled_rng(int(cfg['seed'])*1000003+epoch*37+2601, raw.device):
                fused = model.fuse_nodes(z_bridge, graph_t)
            fused_leaf = fused.detach().requires_grad_(True)

            total_loss = 0.
            for ps in slices(len(pairs), int(cfg['pair_batch'])):
                chunk = pairs[ps]
                base_chunk = base_leaf[chunk[:, 0], chunk[:, 1]]
                predicted, _, _ = model.decode_pairs(fused_leaf, chunk, base_chunk,
                                                     backbone.gene_scale, context)
                loss = (predicted - targets[ps]).square().sum() / (len(pairs) * loss_scale)
                if not bool(torch.isfinite(loss)):
                    raise FloatingPointError('V46 联合 Count MSE 出现 NaN/Inf')
                loss.backward()
                total_loss += float(loss.detach())

            if fused_leaf.grad is None or base_leaf.grad is None:
                raise RuntimeError('梯度未传入 GAT 或 AE 基础解码分支，请检查断图')
            # 先将两条分支的梯度合并到 128 维表示（不保留块级计算图）。
            base_matrix.backward(base_leaf.grad)
            fused.backward(fused_leaf.grad)
            if z_bridge.grad is None:
                raise RuntimeError('128 维特征没有收到来自图与解码器的梯度')
            z.backward(z_bridge.grad)
            if backbone.ae.deep[-1].weight.grad is None or backbone.ae.decoder[0].weight.grad is None:
                raise RuntimeError('瓶颈或解码器没有梯度：联合微调未生效')
            if any(p.grad is not None for p in backbone.ae.shallow.parameters()):
                raise AssertionError('512 浅层不应收到梯度')
            torch.nn.utils.clip_grad_norm_(trainable, float(cfg['grad_clip']), error_if_nonfinite=True)
            optimizer.step()

            history.append(dict(epoch=epoch, scaled_MSE=total_loss))
            if epoch == 1 or epoch % int(cfg['log_every']) == 0 or epoch == int(cfg['correction_epochs']):
                print(f'V46 联合训练 {epoch}/{cfg["correction_epochs"]} | scaled Count MSE {total_loss:.6f}')
            del task, context, z, z_bridge, base_matrix, base_leaf, fused, fused_leaf

        # 检验更新的是深层，浅层没有变化；不是只把参数设为可训练却断了梯度。
        if frozen_model_hash(backbone.ae.shallow) != original_shallow:
            raise AssertionError('构图浅层意外改变！')
        status = dict(
            shallow_frozen=True,
            bottleneck_updated=(frozen_model_hash(backbone.ae.deep) != original_deep),
            decoder_updated=(frozen_model_hash(backbone.ae.decoder) != original_decoder),
        )
        print('联合微调权重自检：', status)
        if not (status['bottleneck_updated'] and status['decoder_updated']):
            raise AssertionError('瓶颈/解码器参数没有更新，请检查梯度和学习率')
        model.eval().requires_grad_(False)
        backbone.ae.eval().requires_grad_(False)
        prediction = v46_predict(raw, backbone, model, graph_t, neighbors, cfg)
    return dict(prediction=prediction, history=pd.DataFrame(history), update_status=status)


def v46_fit_pipeline(raw, cfg):
    with comparison_cpu_scope(raw.device, cfg):
        base = v46_prepare_base(raw, cfg)
        reader = v46_train_joint_reader(raw, cfg, base)
    return dict(prediction=reader['prediction'], graph_hash=base['graph_hash'],
                graph_status=base['graph_status'], ae_history=base['ae_history'],
                reader_history=reader['history'], joint_update=reader['update_status'])


def v46_holdout_validation(counts, seed=42, fraction=.06):
    original = np.asarray(counts, dtype=np.float32)
    if original.ndim != 2 or not np.isfinite(original).all() or (original < 0).any():
        raise ValueError('输入必须是非负且有限的 Count 矩阵')
    rng = np.random.default_rng(int(seed)+38000)
    held = (original > 0) & (rng.random(original.shape) < float(fraction))
    if not held.any():
        raise ValueError('未抽中人工留出非零位置')
    train = original.copy()
    train[held] = 0.
    return train, held


def v46_metrics_on_mask(pred, reference, mask):
    y = np.asarray(reference)[mask].astype(np.float64)
    p = np.asarray(pred)[mask].astype(np.float64)
    pcc = float(np.corrcoef(p,y)[0,1]) if len(y)>1 and p.std()>0 and y.std()>0 else np.nan
    return dict(PCC=pcc, L1=float(np.mean(np.abs(p-y))),
                RMSE=float(np.sqrt(np.mean((p-y)**2))))


## V46：6% 人工隐藏内部验证 → 完整 dropout40 重训（只运行 V46）

In [ ]:
from pathlib import Path
from IPython.display import display

V46_train_counts, V46_holdout = v46_holdout_validation(count_array,seed=42,fraction=.06)
print('人工隐藏验证位置:',int(V46_holdout.sum()))
# A 组历史结果，不重复训练，也绝不用于训练模型或模型选择。
HISTORICAL_A = dict(方案='A_V44_CountGraph_OwnOnly_历史', PCC=0.946109, L1=1.185628, RMSE=8.700918)

# 第一次训练：6% 外层验证位置在所有模型构建之前就已清零。
V46_train_t=torch.as_tensor(np.ascontiguousarray(V46_train_counts),dtype=torch.float32,device=model_device)
torch.manual_seed(42); np.random.seed(42)
V46_validation_run = v46_fit_pipeline(V46_train_t,V46_CONFIG)
V46_validation_filter = v32_build_filter(V46_train_counts,V46_FILTER)
V46_validation_pred = v32_filter_prediction(
    V46_train_counts, np.asarray(V46_validation_run['prediction'],dtype=np.float32),
    V46_validation_filter['masks']['neighbor'])
assert np.array_equal(V46_validation_pred[V46_train_counts>0],V46_train_counts[V46_train_counts>0])
V46_validation_metrics = v46_metrics_on_mask(V46_validation_pred,count_array,V46_holdout)
print('\n=== V46 方案：独立人工遮盖验证（PCC / L1 / RMSE）===')
display(pd.DataFrame([dict(方案='V46_瓶颈解码器联合微调_GAT_GeneRead',**V46_validation_metrics)]))

# 第二次训练：使用完整 dropout40，仍没有读取 true_data.csv。
V46_full_t=torch.as_tensor(np.ascontiguousarray(count_array),dtype=torch.float32,device=model_device)
torch.manual_seed(42); np.random.seed(42)
V46_full_run=v46_fit_pipeline(V46_full_t,V46_CONFIG)
V46_full_filter=v32_build_filter(count_array,V46_FILTER)
V46_raw_pred=np.asarray(V46_full_run['prediction'],dtype=np.float32)
final_imputed=v32_filter_prediction(count_array,V46_raw_pred,V46_full_filter['masks']['neighbor'])
imputed_data=final_imputed
assert np.array_equal(final_imputed[count_array>0],count_array[count_array>0])
assert np.isfinite(final_imputed).all() and (final_imputed>=0).all()
print('V46 方案训练完成：final_imputed 为原始 Count。接下来才读取真实完整参考评价。')


## 最终评价：历史 V44 / V45 与本次 V46（只读一次真实 Count）

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
assert 'final_imputed' in globals(), '必须先完成 V46 方案所有训练和预测'
DATA_DIRS=[Path(r'E:/code/my/zeisel_data'), Path.cwd(), Path.cwd()/'zeisel_data',Path('/mnt/data')]
def find_path(filename):
    for d in DATA_DIRS:
        p=d/filename
        if p.is_file(): return p
    raise FileNotFoundError(filename+' not found')
TRUE_PATH=find_path('true_data.csv')
DROPOUT_PATH=find_path('Zeisel_dropout_40_gene_cell.csv')
print('V46 方案预测已固定；历史 V44 / V45 不会重新训练。')
print('现在才读取 true_data.csv 进行报告:',TRUE_PATH)


In [ ]:
true_frame=pd.read_csv(TRUE_PATH,index_col=0)
drop_frame=pd.read_csv(DROPOUT_PATH,index_col=0)
assert true_frame.index.equals(drop_frame.index) and true_frame.columns.equals(drop_frame.columns)
truth=np.asarray(true_frame.values.T,dtype=np.float64)
raw=np.asarray(drop_frame.values.T,dtype=np.float64)
assert truth.shape==raw.shape==count_array.shape
assert np.array_equal(raw.astype(np.float32),count_array.astype(np.float32))

def final_three_metrics(pred,reference):
    p=np.asarray(pred,dtype=np.float64).ravel()
    y=np.asarray(reference,dtype=np.float64).ravel()
    pcc=float(np.corrcoef(p,y)[0,1]) if p.std()>0 and y.std()>0 else np.nan
    return dict(PCC=pcc,L1=float(np.mean(np.abs(p-y))),RMSE=float(np.sqrt(np.mean((p-y)**2))))

V46_final_metrics=final_three_metrics(final_imputed,truth)
comparison=pd.DataFrame([HISTORICAL_A,
    dict(方案='V45_共享AE128_冻结_历史',PCC=0.941240,L1=1.169234,RMSE=9.055609),
    dict(方案='V46_共享AE后半段联合微调_本次',**V46_final_metrics)],
                        columns=['方案','PCC','L1','RMSE'])
print('=== 历史 V44 / V45 与本次 V46：完整 Count 矩阵（仅 PCC / L1 / RMSE）===')
display(comparison)
print('说明：V44 / V45 均为历史指标，并非本次重训或严格配对的同轮实验。')
